# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

A page is prioritized for review when it had meaningful Google Search visibility
during the historical feature window and its impressions were already declining
compared with the preceding 30-day period.

The baseline ranks these pages by their historical impression volume, so pages
with more search exposure are placed higher in the review queue.

In plain words:

> Review pages that are both visible and already showing a meaningful decline,
> with higher-exposure pages reviewed first.

This is a transparent decision-support baseline, not a prediction model.


---


### Rule components

- **Visible:** at least 100 GSC impressions during March 2026.
- **Declining:** March 2026 impressions are at least 20% below February 2026.
- **Score:** historical March impressions for pages satisfying both conditions.
- Pages that do not satisfy both conditions receive a score of 0.

The baseline does not use April 2026 outcomes when calculating the score.


---


### Reason code

The baseline uses one primary reason code:

- `declining_and_visible` — the page had meaningful historical search visibility
  and its impressions had declined by more than 20% before the prediction moment.

Pages with score 0 receive:

- `not_prioritized`


---


### Action

- `review` — pages satisfying the baseline rule.
- `monitor` — pages that do not satisfy the baseline rule.

In [1]:
%pip -q install duckdb huggingface_hub

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your Hugging Face READ token (hf_...): ··········


### Signal checks

Check two signals first, with bucket tables and n.

I would use:

> Historical impression volume → flag-linked signal  
> Historical impression decline → directly relevant to the lane

#### Signal 1 — Historical impression volume

Volume is useful because a decline on a page with meaningful search visibility
creates a larger potential review opportunity than a decline on a page with
almost no search exposure.

``Verdict: CONFIRMED``

The warehouse contains measurable GSC impressions, and the March 2026 slice
contains pages with substantially different levels of search exposure.

The baseline therefore uses impression volume as its ranking magnitude.

In [2]:
volume_buckets = con.sql(f"""
    WITH page_volume AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_march
        FROM {TABLES["fact_daily"]}
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-04-01'
          AND gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        CASE
            WHEN impressions_march < 10 THEN '<10'
            WHEN impressions_march < 100 THEN '10-99'
            WHEN impressions_march < 500 THEN '100-499'
            WHEN impressions_march < 1000 THEN '500-999'
            WHEN impressions_march < 5000 THEN '1,000-4,999'
            ELSE '5,000+'
        END AS impression_bucket,

        COUNT(*) AS n,

        ROUND(AVG(impressions_march), 1) AS avg_impressions

    FROM page_volume

    GROUP BY 1

    ORDER BY
        CASE impression_bucket
            WHEN '<10' THEN 1
            WHEN '10-99' THEN 2
            WHEN '100-499' THEN 3
            WHEN '500-999' THEN 4
            WHEN '1,000-4,999' THEN 5
            WHEN '5,000+' THEN 6
        END
""").df()

volume_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,avg_impressions
0,<10,33532,3.4
1,10-99,41765,41.8
2,100-499,39517,250.0
3,500-999,16866,716.7
4,"1,000-4,999",31766,2391.8
5,"5,000+",13292,13605.9


#### Signal 2 - Historical decline

To construct the baseline honestly, we need February vs March.  

Pages with fewer than 100 impressions in the previous 30-day period are excluded from the decline signal because percentage changes at very low volume can be noisy. The 100-impression threshold is a practical minimum-volume filter, not a claim that observations below 100 are inherently unreliable.

In [3]:
decline_signal = con.sql(f"""
    WITH monthly AS (

        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date >= DATE '2026-02-01'
                     AND report_date < DATE '2026-03-01'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS impressions_prev30,

            SUM(
                CASE
                    WHEN report_date >= DATE '2026-03-01'
                     AND report_date < DATE '2026-04-01'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS impressions_march

        FROM {TABLES["fact_daily"]}

        WHERE report_date >= DATE '2026-02-01'
          AND report_date < DATE '2026-04-01'

          AND gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        *,
        CASE
            WHEN impressions_prev30 > 0
            THEN
                (impressions_march - impressions_prev30)
                / impressions_prev30
            ELSE NULL
        END AS change_pct

    FROM monthly

    WHERE impressions_prev30 >= 100
""").df()

decline_signal.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_prev30,impressions_march,change_pct
0,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,255.0,0.085106
1,client_3ffa76342f366962,content_456ab2db28595187,100.0,49.0,-0.510000
2,client_3ffa76342f366962,content_5573434837db89c5,198.0,90.0,-0.545455
3,client_3ffa76342f366962,content_7b17975c58745266,102.0,16.0,-0.843137
4,client_3ffa76342f366962,content_b89167cd03d6ffc1,178.0,68.0,-0.617978



The decline signal compares March 2026 impressions with February 2026
impressions.

```Verdict: CONFIRMED```

The warehouse contains enough historical GSC data to calculate a directional
month-over-month decline signal for pages with meaningful prior exposure.

The threshold used by this baseline is a decline greater than 20%.

In [4]:
decline_buckets = pd.DataFrame({
    "decline_bucket": [
        "< -50%",
        "-50% to -20%",
        "-20% to 0%",
        "0% to +20%",
        "> +20%"
    ],
    "n": [
        (decline_signal["change_pct"] < -0.50).sum(),
        (
            (decline_signal["change_pct"] >= -0.50)
            & (decline_signal["change_pct"] < -0.20)
        ).sum(),
        (
            (decline_signal["change_pct"] >= -0.20)
            & (decline_signal["change_pct"] < 0)
        ).sum(),
        (
            (decline_signal["change_pct"] >= 0)
            & (decline_signal["change_pct"] <= 0.20)
        ).sum(),
        (decline_signal["change_pct"] > 0.20).sum()
    ]
})

decline_buckets

,decline_bucket,n
0,< -50%,8411
1,-50% to -20%,9063
2,-20% to 0%,9896
3,0% to +20%,10857
4,> +20%,42095


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

```text
February 2026 ─────── March 2026 ─────── April 2026
      │                     │                   │
      │                     │                   │
      └── baseline input ───┘                   │
                            │                   │
                       decision point           │
                                                │
                                     evaluation label only

In [5]:
baseline = decline_signal.copy()

# Meaningful visibility in the feature window
baseline["visible"] = (
    baseline["impressions_march"] >= 100
).astype(int)

# Historical decline > 20%
baseline["declining"] = (
    baseline["change_pct"] < -0.20
).astype(int)

# Transparent rule:
# visible AND declining -> rank by historical impression exposure
baseline["baseline_score"] = (
    baseline["visible"]
    * baseline["declining"]
    * baseline["impressions_march"]
)

baseline["reason_code"] = np.where(
    baseline["baseline_score"] > 0,
    "declining_and_visible",
    "not_prioritized"
)

baseline["action"] = np.where(
    baseline["baseline_score"] > 0,
    "review",
    "monitor"
)

baseline = baseline.sort_values(
    ["baseline_score", "impressions_march"],
    ascending=False
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

baseline.head(20)

,client_hash_id,content_hash_id,impressions_prev30,impressions_march,change_pct,visible,declining,baseline_score,reason_code,action,rank
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,134984.0,-0.336365,1,1,134984.0,declining_and_visible,review,1
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,124075.0,-0.360286,1,1,124075.0,declining_and_visible,review,2
2,client_73cda7b4e4f265ea,content_db122b8ba22641b8,127941.0,91408.0,-0.285546,1,1,91408.0,declining_and_visible,review,3
3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,83834.0,-0.571506,1,1,83834.0,declining_and_visible,review,4
4,client_73cda7b4e4f265ea,content_c19eed2225ee5f40,98333.0,72918.0,-0.258459,1,1,72918.0,declining_and_visible,review,5
5,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,113798.0,65736.0,-0.422345,1,1,65736.0,declining_and_visible,review,6
6,client_e547b89c05043229,content_c9a0c2fdbdbfb562,142215.0,65681.0,-0.538157,1,1,65681.0,declining_and_visible,review,7
7,client_62f4a7e64f5e0096,content_abaf75df8fc88085,79649.0,62223.0,-0.218785,1,1,62223.0,declining_and_visible,review,8
8,client_73cda7b4e4f265ea,content_29c4a3831609805d,129662.0,60919.0,-0.530171,1,1,60919.0,declining_and_visible,review,9
9,client_73cda7b4e4f265ea,content_0c6f1068219a8d16,83873.0,58541.0,-0.302028,1,1,58541.0,declining_and_visible,review,10


In [6]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions_prev30",
    "impressions_march",
    "change_pct",
    "baseline_score",
    "reason_code",
    "action"
]

baseline[output_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(
    f"Wrote {len(baseline):,} rows to "
    "work/outputs/baseline_action_score.csv"
)

Wrote 80,322 rows to work/outputs/baseline_action_score.csv


In [7]:
queue = pd.read_csv(
    "work/outputs/baseline_action_score.csv"
)

queue.head(20)

,rank,client_hash_id,content_hash_id,impressions_prev30,impressions_march,change_pct,baseline_score,reason_code,action
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,134984.0,-0.336365,134984.0,declining_and_visible,review
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,124075.0,-0.360286,124075.0,declining_and_visible,review
2,3,client_73cda7b4e4f265ea,content_db122b8ba22641b8,127941.0,91408.0,-0.285546,91408.0,declining_and_visible,review
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,83834.0,-0.571506,83834.0,declining_and_visible,review
4,5,client_73cda7b4e4f265ea,content_c19eed2225ee5f40,98333.0,72918.0,-0.258459,72918.0,declining_and_visible,review
5,6,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,113798.0,65736.0,-0.422345,65736.0,declining_and_visible,review
6,7,client_e547b89c05043229,content_c9a0c2fdbdbfb562,142215.0,65681.0,-0.538157,65681.0,declining_and_visible,review
7,8,client_62f4a7e64f5e0096,content_abaf75df8fc88085,79649.0,62223.0,-0.218785,62223.0,declining_and_visible,review
8,9,client_73cda7b4e4f265ea,content_29c4a3831609805d,129662.0,60919.0,-0.530171,60919.0,declining_and_visible,review
9,10,client_73cda7b4e4f265ea,content_0c6f1068219a8d16,83873.0,58541.0,-0.302028,58541.0,declining_and_visible,review


#### Evaluate Precision@K

The baseline needs to be evaluated against the same future label that **the ML-04 model** will eventually use.




In [8]:
future_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_next30

    FROM {TABLES["fact_daily"]}

    WHERE report_date >= DATE '2026-04-01'
      AND report_date < DATE '2026-05-01'

      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

future_outcomes.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_next30
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0


In [9]:
evaluation = baseline.merge(
    future_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

evaluation["is_declining"] = (
    evaluation["imp_next30"]
    < 0.8 * evaluation["impressions_march"]
).astype(int)

print(
    f"Evaluation pages: {len(evaluation):,}"
)

print(
    f"Future decline base rate: "
    f"{evaluation['is_declining'].mean():.3f}"
)

Evaluation pages: 76,166
Future decline base rate: 0.547


#### Precision@K Function

The baseline prioritizes pages using historical visibility and historical decline. ``Precision@K`` measures how often the pages at the top of that historical rule-based queue subsequently experienced the defined decline.

In [10]:
def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    top_k = labels[order[:k]]

    return top_k.mean()

In [11]:
base_rate = evaluation["is_declining"].mean()

print(f"Future decline base rate: {base_rate:.3f}")
print()

for k in [20, 50, 100]:
    p_at_k = precision_at_k(
        evaluation["baseline_score"],
        evaluation["is_declining"],
        k)

    print(f"Baseline Precision@{k}: {p_at_k:.3f}")

Future decline base rate: 0.547

Baseline Precision@20: 0.400
Baseline Precision@50: 0.360
Baseline Precision@100: 0.450


Future decline base rate: `54.7%`. The baseline achieved `Precision@20` of `40.0%`, `Precision@50` of `36.0%`, and `Precision@100` of `45.0%`. Therefore, the current rule-based ranking does not beat the base-rate benchmark.

#### Dummy baseline

In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

X_dummy = np.zeros((len(evaluation), 1))

y_eval = evaluation["is_declining"]

dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_dummy, y_eval)

dummy_pred = dummy.predict(X_dummy)

print("Dummy accuracy:", round(accuracy_score(y_eval, dummy_pred), 3))

print("Future decline base rate:", round(y_eval.mean(), 3))

Dummy accuracy: 0.547
Future decline base rate: 0.547


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top-20 pages are reviewed as a decision-support queue.

For each page I record:

- the proposed action;
- the reason code;
- a confidence note based only on the historical signals used by the rule;
- what could make the recommendation wrong.

The confidence note is not a calibrated probability. It only describes how
strongly the observed historical signals satisfy the rule.

In [14]:
top20 = evaluation.sort_values(
    "baseline_score",
    ascending=False
).head(20).copy()

top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions_prev30",
        "impressions_march",
        "change_pct",
        "baseline_score",
        "reason_code",
        "action",
        "imp_next30",
        "is_declining"
    ]
]

,rank,client_hash_id,content_hash_id,impressions_prev30,impressions_march,change_pct,baseline_score,reason_code,action,imp_next30,is_declining
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,134984.0,-0.336365,134984.0,declining_and_visible,review,128092.0,0
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,124075.0,-0.360286,124075.0,declining_and_visible,review,73725.0,1
2,3,client_73cda7b4e4f265ea,content_db122b8ba22641b8,127941.0,91408.0,-0.285546,91408.0,declining_and_visible,review,71495.0,1
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,83834.0,-0.571506,83834.0,declining_and_visible,review,188.0,1
4,5,client_73cda7b4e4f265ea,content_c19eed2225ee5f40,98333.0,72918.0,-0.258459,72918.0,declining_and_visible,review,73029.0,0
5,6,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,113798.0,65736.0,-0.422345,65736.0,declining_and_visible,review,55209.0,0
6,7,client_e547b89c05043229,content_c9a0c2fdbdbfb562,142215.0,65681.0,-0.538157,65681.0,declining_and_visible,review,77451.0,0
7,8,client_62f4a7e64f5e0096,content_abaf75df8fc88085,79649.0,62223.0,-0.218785,62223.0,declining_and_visible,review,72578.0,0
8,9,client_73cda7b4e4f265ea,content_29c4a3831609805d,129662.0,60919.0,-0.530171,60919.0,declining_and_visible,review,54726.0,0
9,10,client_73cda7b4e4f265ea,content_0c6f1068219a8d16,83873.0,58541.0,-0.302028,58541.0,declining_and_visible,review,52451.0,0


In [15]:
def confidence_note(row):
    decline = row["change_pct"]
    volume = row["impressions_march"]

    if decline <= -0.50 and volume >= 1000:
        return "Strong historical signal: large decline and high exposure."

    elif decline <= -0.50:
        return "Strong historical decline, but exposure is more limited."

    elif decline <= -0.20 and volume >= 1000:
        return "Moderate historical decline with substantial exposure."

    else:
        return "Meets the rule, but the historical decline is closer to the threshold."


def wrong_if(row):
    decline = row["change_pct"]
    volume = row["impressions_march"]

    if volume < 200:
        return (
            "Could be wrong if the observed decline is driven by low-volume "
            "search noise."
        )

    elif decline > -0.30:
        return (
            "Could be wrong if the modest decline is temporary or normal "
            "month-to-month variation."
        )

    elif decline <= -0.50:
        return (
            "Could be wrong if the large decline reflects seasonality, "
            "external search-demand changes, or another temporary factor."
        )

    else:
        return (
            "Could be wrong if the historical decline does not continue "
            "into the future."
        )


top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

top20[
    [
        "rank",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,review,declining_and_visible,Moderate historical decline with substantial e...,Could be wrong if the historical decline does ...
1,2,review,declining_and_visible,Moderate historical decline with substantial e...,Could be wrong if the historical decline does ...
2,3,review,declining_and_visible,Moderate historical decline with substantial e...,Could be wrong if the modest decline is tempor...
3,4,review,declining_and_visible,Strong historical signal: large decline and hi...,Could be wrong if the large decline reflects s...
4,5,review,declining_and_visible,Moderate historical decline with substantial e...,Could be wrong if the modest decline is tempor...
5,6,review,declining_and_visible,Moderate historical decline with substantial e...,Could be wrong if the historical decline does ...
6,7,review,declining_and_visible,Strong historical signal: large decline and hi...,Could be wrong if the large decline reflects s...
7,8,review,declining_and_visible,Moderate historical decline with substantial e...,Could be wrong if the modest decline is tempor...
8,9,review,declining_and_visible,Strong historical signal: large decline and hi...,Could be wrong if the large decline reflects s...
9,10,review,declining_and_visible,Moderate historical decline with substantial e...,Could be wrong if the historical decline does ...


In [16]:
review_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "impressions_march",
    "change_pct",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20[review_columns]

,rank,client_hash_id,content_hash_id,action,reason_code,impressions_march,change_pct,confidence_note,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,review,declining_and_visible,134984.0,-0.336365,Moderate historical decline with substantial e...,Could be wrong if the historical decline does ...
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,review,declining_and_visible,124075.0,-0.360286,Moderate historical decline with substantial e...,Could be wrong if the historical decline does ...
2,3,client_73cda7b4e4f265ea,content_db122b8ba22641b8,review,declining_and_visible,91408.0,-0.285546,Moderate historical decline with substantial e...,Could be wrong if the modest decline is tempor...
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,review,declining_and_visible,83834.0,-0.571506,Strong historical signal: large decline and hi...,Could be wrong if the large decline reflects s...
4,5,client_73cda7b4e4f265ea,content_c19eed2225ee5f40,review,declining_and_visible,72918.0,-0.258459,Moderate historical decline with substantial e...,Could be wrong if the modest decline is tempor...
5,6,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,review,declining_and_visible,65736.0,-0.422345,Moderate historical decline with substantial e...,Could be wrong if the historical decline does ...
6,7,client_e547b89c05043229,content_c9a0c2fdbdbfb562,review,declining_and_visible,65681.0,-0.538157,Strong historical signal: large decline and hi...,Could be wrong if the large decline reflects s...
7,8,client_62f4a7e64f5e0096,content_abaf75df8fc88085,review,declining_and_visible,62223.0,-0.218785,Moderate historical decline with substantial e...,Could be wrong if the modest decline is tempor...
8,9,client_73cda7b4e4f265ea,content_29c4a3831609805d,review,declining_and_visible,60919.0,-0.530171,Strong historical signal: large decline and hi...,Could be wrong if the large decline reflects s...
9,10,client_73cda7b4e4f265ea,content_0c6f1068219a8d16,review,declining_and_visible,58541.0,-0.302028,Moderate historical decline with substantial e...,Could be wrong if the historical decline does ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks are pages that technically satisfy the rule but have either
relatively low historical exposure or a decline close to the `-20%` threshold.

These are useful stress cases because the rule is deliberately simple. A page
can satisfy the rule without the future decline actually continuing.

The baseline should therefore be interpreted as a review-prioritization rule,
not as proof that a page will decline.

In [17]:
weak_picks = top20[
    (top20["change_pct"] > -0.30)
    |
    (top20["impressions_march"] < 200)
]

weak_picks[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions_march",
        "change_pct",
        "baseline_score",
        "reason_code"
    ]
]

,rank,client_hash_id,content_hash_id,impressions_march,change_pct,baseline_score,reason_code
2,3,client_73cda7b4e4f265ea,content_db122b8ba22641b8,91408.0,-0.285546,91408.0,declining_and_visible
4,5,client_73cda7b4e4f265ea,content_c19eed2225ee5f40,72918.0,-0.258459,72918.0,declining_and_visible
7,8,client_62f4a7e64f5e0096,content_abaf75df8fc88085,62223.0,-0.218785,62223.0,declining_and_visible
10,11,client_23a62021009f63c4,content_f7c9fcc26f6e23c1,57781.0,-0.255256,57781.0,declining_and_visible
11,12,client_73cda7b4e4f265ea,content_e8b074fd4a082388,56446.0,-0.205870,56446.0,declining_and_visible
12,13,client_62f4a7e64f5e0096,content_3e07fb36c28b980e,56296.0,-0.223268,56296.0,declining_and_visible
13,14,client_62f4a7e64f5e0096,content_b03d1bccd56145c0,49407.0,-0.218578,49407.0,declining_and_visible
15,16,client_73cda7b4e4f265ea,content_d30a67f972196ca1,46810.0,-0.294573,46810.0,declining_and_visible
17,18,client_73cda7b4e4f265ea,content_c79d24a701ffcd6c,44918.0,-0.204949,44918.0,declining_and_visible
18,19,client_73cda7b4e4f265ea,content_34e3f342bfd1dbf4,42927.0,-0.277737,42927.0,declining_and_visible


In [18]:
baseline_feature_columns = [
    "impressions_prev30",
    "impressions_march",
    "change_pct"
]

future_or_label_columns = [
    "imp_next30",
    "is_declining"
]

print("Baseline score inputs:")
for col in baseline_feature_columns:
    print("  ", col)

print("\nFuture/label columns:")
for col in future_or_label_columns:
    print("  ", col)

Baseline score inputs:
   impressions_prev30
   impressions_march
   change_pct

Future/label columns:
   imp_next30
   is_declining


In [19]:
expected_score = (
    (evaluation["impressions_march"] >= 100).astype(int)
    *
    (evaluation["change_pct"] < -0.20).astype(int)
    *
    evaluation["impressions_march"]
)

print(
    "Score matches rule:",
    np.allclose(
        evaluation["baseline_score"],
        expected_score
    )
)

Score matches rule: True


→ Implementation matches the documented rule.

In [20]:
score_check = evaluation[
    [
        "baseline_score",
        "impressions_prev30",
        "impressions_march",
        "change_pct",
        "imp_next30",
        "is_declining"
    ]
].head(10)

score_check

,baseline_score,impressions_prev30,impressions_march,change_pct,imp_next30,is_declining
0,134984.0,203401.0,134984.0,-0.336365,128092.0,0
1,124075.0,193954.0,124075.0,-0.360286,73725.0,1
2,91408.0,127941.0,91408.0,-0.285546,71495.0,1
3,83834.0,195648.0,83834.0,-0.571506,188.0,1
4,72918.0,98333.0,72918.0,-0.258459,73029.0,0
5,65736.0,113798.0,65736.0,-0.422345,55209.0,0
6,65681.0,142215.0,65681.0,-0.538157,77451.0,0
7,62223.0,79649.0,62223.0,-0.218785,72578.0,0
8,60919.0,129662.0,60919.0,-0.530171,54726.0,0
9,58541.0,83873.0,58541.0,-0.302028,52451.0,0


```text
                BASELINE SCORE
                     │
        ┌────────────┼────────────┐
        ↓            ↓            ↓
 February        March       March vs Feb
 impressions   impressions     change
        │            │            │
        └────────────┼────────────┘
                     ↓
              baseline_score
                     │
                     │
             prediction moment
                     │
                     ▼
                 April 2026
                     │
                     ▼
              is_declining
              EVALUATION ONLY

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.